# E2B Example: Parallel Data Science Workbench

This example splits one dataset task across three isolated E2B sandboxes:

- `eda_runner`: profile the data and highlight anomalies
- `cleaning_runner`: propose cleaning steps and emit a cleaned artifact
- `dashboard_runner`: build a tiny previewable data app


In [ ]:
import importlib
import json
import os
import subprocess
import sys
from pathlib import Path

repo_root = Path.cwd()
if not (repo_root / "src" / "agents").exists():
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "src" / "agents").exists():
            repo_root = candidate
            break
        if (candidate / "openai-agents-python-preview" / "src" / "agents").exists():
            repo_root = candidate / "openai-agents-python-preview"
            break

src_root = repo_root / "src"
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
if str(src_root) not in sys.path:
    sys.path.insert(0, str(src_root))

env_path = None
for candidate in [Path.cwd(), *Path.cwd().parents, repo_root, *repo_root.parents]:
    maybe_env = candidate / ".env"
    if maybe_env.exists():
        env_path = maybe_env
        break

if env_path is not None:
    try:
        from dotenv import load_dotenv

        load_dotenv(env_path, override=False)
    except Exception:
        for raw_line in env_path.read_text().splitlines():
            line = raw_line.strip()
            if not line or line.startswith("#") or "=" not in line:
                continue
            key, value = line.split("=", 1)
            os.environ.setdefault(key.strip(), value.strip().strip('"').strip("'"))

required_modules = ["agents", "openai", "e2b", "e2b_code_interpreter"]
missing_modules = []
for module_name in required_modules:
    try:
        importlib.import_module(module_name)
    except Exception:
        missing_modules.append(module_name)

if missing_modules:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-e", f"{repo_root}[e2b]"])

print(f"Using repo root: {repo_root}")

In [ ]:
from pydantic import BaseModel, Field

from agents import Agent, ModelSettings, Runner
from agents.extensions.sandbox import (
    E2BSandboxClient,
    E2BSandboxClientOptions,
    E2BSandboxType,
)
from agents.run import RunConfig
from agents.sandbox import SandboxAgent, SandboxRunConfig
from examples.sandbox.misc.example_support import text_manifest
from examples.sandbox.misc.workspace_shell import WorkspaceShellCapability

MODEL = os.getenv("OPENAI_MODEL", "gpt-5.4")
SANDBOX_TYPE = E2BSandboxType(os.getenv("E2B_SANDBOX_TYPE", E2BSandboxType.E2B.value))
TEMPLATE = os.getenv("E2B_TEMPLATE") or None


def make_e2b_run_config(*, exposed_ports: tuple[int, ...] = ()) -> RunConfig:
    return RunConfig(
        sandbox=SandboxRunConfig(
            client=E2BSandboxClient(),
            options=E2BSandboxClientOptions(
                sandbox_type=SANDBOX_TYPE,
                template=TEMPLATE,
                timeout=300,
                exposed_ports=exposed_ports,
            ),
        )
    )


class EDAResult(BaseModel):
    summary: str
    anomalies: list[str]
    evidence_files: list[str] = Field(min_length=1)


class CleaningResult(BaseModel):
    summary: str
    cleaned_artifact: str
    transformations: list[str]
    evidence_files: list[str] = Field(min_length=1)


class DashboardResult(BaseModel):
    summary: str
    app_port: int
    evidence_files: list[str] = Field(min_length=1)

In [ ]:
dataset_csv = (
    "month,revenue,signups,churn\n"
    "2026-01,120000,210,0.04\n"
    "2026-02,125000,220,0.03\n"
    "2026-03,99000,140,0.08\n"
    "2026-04,132000,240,0.03\n"
)

manifest = text_manifest(
    {
        "metrics.csv": dataset_csv,
        "README.md": "Use metrics.csv for analysis.\n",
        "dashboard.html": "<html><body><h1>Metrics Dashboard</h1></body></html>",
    }
)

eda_runner = SandboxAgent(
    name="EDA Runner",
    model=MODEL,
    instructions=("Inspect the dataset, identify anomalies, and return a structured EDA summary."),
    developer_instructions=("Use the shell tool before answering. Only cite inspected files."),
    default_manifest=manifest,
    capabilities=[WorkspaceShellCapability()],
    model_settings=ModelSettings(tool_choice="required"),
    output_type=EDAResult,
)

cleaning_runner = SandboxAgent(
    name="Cleaning Runner",
    model=MODEL,
    instructions=(
        "Inspect the dataset, propose cleaning steps, and pretend to write a cleaned artifact."
    ),
    developer_instructions=(
        "Use the shell tool. You may write cleaned_metrics.csv before answering."
    ),
    default_manifest=manifest,
    capabilities=[WorkspaceShellCapability()],
    model_settings=ModelSettings(tool_choice="required"),
    output_type=CleaningResult,
)

dashboard_runner = SandboxAgent(
    name="Dashboard Runner",
    model=MODEL,
    instructions=("Inspect the dataset and start a tiny data dashboard preview on port 8765."),
    developer_instructions=(
        "Use the shell tool. Start a background HTTP server with python -m http.server 8765."
    ),
    default_manifest=manifest,
    capabilities=[WorkspaceShellCapability()],
    model_settings=ModelSettings(tool_choice="required"),
    output_type=DashboardResult,
)

In [ ]:
preview_client = E2BSandboxClient()
dashboard_session = await preview_client.create(
    manifest=manifest,
    options=E2BSandboxClientOptions(
        sandbox_type=SANDBOX_TYPE,
        template=TEMPLATE,
        timeout=300,
        exposed_ports=(8765,),
    ),
)
await dashboard_session.start()


async def structured_output_with_preview(result) -> str:
    final_output = result.final_output
    payload = (
        final_output.model_dump(mode="json")
        if isinstance(final_output, BaseModel)
        else {"raw": str(final_output)}
    )
    if payload.get("app_port"):
        endpoint = await dashboard_session.resolve_exposed_port(int(payload["app_port"]))
        payload["preview_url"] = endpoint.url_for("http") + "dashboard.html"
    return json.dumps(payload, sort_keys=True)


orchestrator = Agent(
    name="Data Science Coordinator",
    model=MODEL,
    instructions=(
        "Run the EDA, cleaning, and dashboard lanes. The dashboard lane tool output includes "
        "a `preview_url` field that was resolved from the sandbox session. Use that exact "
        "`preview_url` value in the final answer and never replace it with localhost or an "
        "invented URL. Summarize the most important finding, the cleaned artifact path, and "
        "the dashboard preview URL."
    ),
    model_settings=ModelSettings(tool_choice="required"),
    tools=[
        eda_runner.as_tool(
            tool_name="run_eda_lane",
            tool_description="Run exploratory data analysis in an isolated sandbox.",
            custom_output_extractor=structured_output_with_preview,
            run_config=make_e2b_run_config(),
        ),
        cleaning_runner.as_tool(
            tool_name="run_cleaning_lane",
            tool_description="Run data cleaning analysis in an isolated sandbox.",
            custom_output_extractor=structured_output_with_preview,
            run_config=make_e2b_run_config(),
        ),
        dashboard_runner.as_tool(
            tool_name="run_dashboard_lane",
            tool_description="Run dashboard generation in an isolated sandbox.",
            custom_output_extractor=structured_output_with_preview,
            run_config=RunConfig(sandbox=SandboxRunConfig(session=dashboard_session)),
        ),
    ],
)

result = await Runner.run(orchestrator, "Analyze this metrics dataset in parallel.")  # type: ignore[top-level-await]  # noqa: F704
print(result.final_output)